In [4]:
# ============================================================
# Cell 1: Load data + setup
# ============================================================
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

# ---- ডেটা লোড ----
train_ood = pd.read_csv("../data/processed/train_ood.csv")
test_ood  = pd.read_csv("../data/processed/test_ood.csv")

# যদি campaign split-ও চান:
import os
if os.path.exists("../data/processed/train_campaign.csv"):
    train_camp = pd.read_csv("../data/processed/train_campaign.csv")
    test_camp  = pd.read_csv("../data/processed/test_campaign.csv")

# ---- যাচাই ----
print("train_ood:", train_ood.shape)
print("test_ood :", test_ood.shape)
print("\ntrain label dist:\n", train_ood['label'].value_counts())
print("\ntest label dist:\n",  test_ood['label'].value_counts())
print("\ntrain columns:", train_ood.columns.tolist())

train_ood: (3525, 4)
test_ood : (3480, 4)

train label dist:
 label
smish     1413
normal    1245
promo      867
Name: count, dtype: int64

test label dist:
 label
smish     1396
normal    1243
promo      841
Name: count, dtype: int64

train columns: ['label', 'text', 'source', 'text_clean']


In [5]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "sagorsarker/bangla-bert-base"
tok = AutoTokenizer.from_pretrained(model_name)
enc = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
enc = enc.to(device)
enc.eval()

def get_embeddings(texts, batch=32):
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            batch_texts = [str(t) for t in texts[i:i+batch]]
            inputs = tok(batch_texts, padding=True, truncation=True,
                         max_length=128, return_tensors='pt').to(device)
            out = enc(**inputs).last_hidden_state[:, 0, :]
            embs.append(out.cpu().numpy())
    return np.vstack(embs)

print("Device:", device)
print("Model loaded ✅")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
Model loaded ✅


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report)

print("Embedding train (3,525 SMS)...")
X_tr = get_embeddings(train_ood['text_clean'].tolist())
print("✅ X_tr:", X_tr.shape)

print("Embedding test (3,480 SMS)...")
X_te = get_embeddings(test_ood['text_clean'].tolist())
print("✅ X_te:", X_te.shape)

print("\nTraining Logistic Regression...")
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf.fit(X_tr, train_ood['label'])

preds = clf.predict(X_te)

print("\n" + "=" * 60)
print("BanglaBERT (frozen) + Linear Probe — OOD Test")
print("=" * 60)
print(f"Accuracy : {accuracy_score(test_ood['label'], preds):.4f}")
print(f"Macro F1 : {f1_score(test_ood['label'], preds, average='macro'):.4f}")
print("\n", classification_report(test_ood['label'], preds))

Embedding train (3,525 SMS)...
✅ X_tr: (3525, 768)
Embedding test (3,480 SMS)...
✅ X_te: (3480, 768)

Training Logistic Regression...

BanglaBERT (frozen) + Linear Probe — OOD Test
Accuracy : 0.8437
Macro F1 : 0.8399

               precision    recall  f1-score   support

      normal       0.92      0.80      0.85      1243
       promo       0.84      0.79      0.81       841
       smish       0.80      0.92      0.85      1396

    accuracy                           0.84      3480
   macro avg       0.85      0.84      0.84      3480
weighted avg       0.85      0.84      0.84      3480



: 